# TicketPilot - Sprint 2, Day 8: Fine-Tuning DistilBERT on Google Colab (Free GPU)

This notebook trains an end-to-end transformer model (**DistilBERT**) for support ticket queue triage.

### Instructions:
1. In the menu, go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Run all cells in sequence.

In [ ]:
# 1. Verify GPU acceleration
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU. Switch runtime to T4 GPU for fast training!")

In [ ]:
# 2. Install required libraries
!pip install -q transformers datasets accelerate scikit-learn plotly

In [ ]:
# 3. Upload or Load pre-split train.csv and val.csv
import pandas as pd
import os

# If uploading files manually in Colab:
# from google.colab import files
# uploaded = files.upload()

# Read data splits
train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('val.csv')
print(f"Train size: {len(train_df):,} | Val size: {len(val_df):,}")

In [ ]:
# 4. Setup Labels and Inverse Class Weights
import numpy as np

unique_queues = sorted(train_df['queue'].unique())
label2id = {label: i for i, label in enumerate(unique_queues)}
id2label = {i: label for i, label in enumerate(unique_queues)}

train_labels = [label2id[q] for q in train_df['queue']]
val_labels = [label2id[q] for q in val_df['queue']]

# Compute inverse class weights to counteract class imbalance
class_counts = np.bincount(train_labels, minlength=len(unique_queues))
class_weights = len(train_labels) / (len(unique_queues) * class_counts.astype(np.float32))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Class weights computed for 10 queues.")

In [ ]:
# 5. Tokenize text using DistilBertTokenizer
from transformers import AutoTokenizer

model_ckpt = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

train_encodings = tokenizer(train_df['text'].tolist(), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_df['text'].tolist(), truncation=True, padding=True, max_length=128)

class TicketDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = TicketDataset(train_encodings, train_labels)
val_dataset = TicketDataset(val_encodings, val_labels)

In [ ]:
# 6. Define Custom Weighted Trainer & Metrics
import torch.nn as nn
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weights is not None:
            weight_tensor = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weight_tensor)
        else:
            loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'f1_macro': f1_macro, 'precision_macro': p_macro, 'recall_macro': r_macro}

In [ ]:
# 7. Train with Overfitting Protection (load_best_model_at_end & EarlyStopping)
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=len(unique_queues),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir='./distilbert_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    report_to='none'
)

trainer = WeightedLossTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
print("Training Complete!")

In [ ]:
# 8. Save best fine-tuned model
save_dir = './distilbert_best_model'
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
!zip -r distilbert_model.zip distilbert_best_model
print("Saved and zipped best model. Download 'distilbert_model.zip' and place in your models/ folder!")